In [57]:
# pip install pymatgen
import os
from pymatgen.ext.matproj import MPRester
from pymatgen.analysis.phase_diagram import PhaseDiagram

# 0) Make sure your API key is set:
# export MAPI_KEY="YOUR_KEY_HERE"
api_key = 'pJx41Nd1Bl2zMSeN8JPeo9a65TJG0dAY'
# os.environ.get("pJx41Nd1Bl2zMSeN8JPeo9a65TJG0dAY")

import requests

In [58]:
import pymatgen

In [59]:
pymatgen.ext

<module 'pymatgen.ext' (<_frozen_importlib_external._NamespaceLoader object at 0x10561a7d0>)>

In [60]:
session = requests.Session()
session.verify = False  # ⚠️ insecure, only for debugging

elements = ["Mn", "Cr", "In"]   # <-- your chemistry
with MPRester(api_key) as mpr:
    # 1) Fetch entries with formation energies for the given chemical system
    entries = mpr.get_entries_in_chemsys(elements)

    # 2) Build phase diagram and get stable entries
    pd = PhaseDiagram(entries)
    stable_entries = pd.stable_entries

    # 3) Gather energies and structures
    results = []
    for e in stable_entries:
        # entry_id is typically the MP material_id for ComputedEntry
        mp_id = getattr(e, "data", None)['material_id']
        formula = e.composition.reduced_formula
        E_form_pa = pd.get_form_energy_per_atom(e)          # formation energy/atom
        E_above = pd.get_e_above_hull(e)                    # 0 for hull entries

        # 4) Fetch a representative structure from MP
        structure = None
        if mp_id:
            try:
                structure = mpr.get_structure_by_material_id(mp_id)  # pymatgen Structure
            except Exception:
                pass

        results.append({
            "mp_id": mp_id,
            "formula": formula,
            "E_form_per_atom_eV": E_form_pa,
            "E_above_hull_eV_per_atom": E_above,
            "structure": structure,
        })

In [ ]:
getattr(e, "data", None)['material_id']

In [51]:
e

mp-90-GGA ComputedStructureEntry - Cr2          (Cr)
Energy (Uncorrected)     = -19.3061  eV (-9.6530  eV/atom)
Correction               = 0.0000    eV (0.0000   eV/atom)
Energy (Final)           = -19.3061  eV (-9.6530  eV/atom)
Energy Adjustments:
  None
Parameters:
  potcar_spec            = [{'titel': 'PAW_PBE Cr_pv 07Sep2000', 'hash': 'eb23364cc25164418f9f79efd8f04f7d', 'summary_stats': None}]
  run_type               = GGA
  is_hubbard             = False
  hubbards               = None
Data:
  oxide_type             = None
  aspherical             = True
  last_updated           = 2024-11-21 20:28:09.060145+00:00
  task_id                = mp-1271227
  material_id            = mp-90
  oxidation_states       = {'Cr': 0.0}
  license                = BY-C
  run_type               = GGA

In [75]:
pd.get_hull_energy_per_atom

Signature: pd.get_decomp_and_hull_energy_per_atom(comp: 'Composition') -> 'tuple[dict[PDEntry, float], float]'
Docstring:
Args:
    comp (Composition): Input composition.

Returns:
    Energy of lowest energy equilibrium at desired composition per atom
File:      ~/anaconda3/envs/HTP/lib/python3.10/site-packages/pymatgen/analysis/phase_diagram.py
Type:      method

In [77]:
pd.get_form_energy_per_atom(entries[4])

0.30830919624999975

In [71]:
pd.get_e_above_hull(entries[4])

np.float64(0.30830919624999975)

In [72]:
pd.get_e_above_hull?

Signature: pd.get_e_above_hull(entry: 'PDEntry', **kwargs: 'Any') -> 'float | None'
Docstring:
Provides the energy above convex hull for an entry.

Args:
    entry (PDEntry): A PDEntry like object.
    **kwargs: Passed to get_decomp_and_e_above_hull().

Returns:
    float | None: Energy above convex hull of entry. Stable entries should have
        energy above hull of 0. The energy is given per atom.
File:      ~/anaconda3/envs/HTP/lib/python3.10/site-packages/pymatgen/analysis/phase_diagram.py
Type:      method

In [69]:
entries[4]

mp-1183750-GGA ComputedStructureEntry - Cr2 In6      (CrIn3)
Energy (Uncorrected)     = -33.3497  eV (-4.1687  eV/atom)
Correction               = 0.0000    eV (0.0000   eV/atom)
Energy (Final)           = -33.3497  eV (-4.1687  eV/atom)
Energy Adjustments:
  None
Parameters:
  potcar_spec            = [{'titel': 'PAW_PBE Cr_pv 07Sep2000', 'hash': 'eb23364cc25164418f9f79efd8f04f7d', 'summary_stats': None}, {'titel': 'PAW_PBE In_d 06Sep2000', 'hash': 'b87558aef4b20a3c4a008ff3e8775108', 'summary_stats': None}]
  run_type               = GGA
  is_hubbard             = False
  hubbards               = None
Data:
  oxide_type             = None
  aspherical             = True
  last_updated           = 2024-11-21 20:29:51.501798+00:00
  task_id                = mp-1829629
  material_id            = mp-1183750
  oxidation_states       = {}
  license                = BY-C
  run_type               = GGA

In [ ]:
e